This is for quickly fetch lots of data for analysis study only

In [10]:
import pandas as pd
import numpy as np

import time
from datetime import datetime, timedelta

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException

import os
from sqlalchemy import create_engine

In [11]:
def get_data(card, css, attr="innerText"):
    try:
        return card.find_element(By.CSS_SELECTOR, css).get_attribute(attr)
    except NoSuchElementException:
        return None

In [ ]:
def run(t):
    for i in range(t):
        checkin_date = datetime.now() + timedelta(days=14+i)
        checkout_date = checkin_date + timedelta(days=1)
        checkin_str = checkin_date.strftime('%Y-%m-%d')
        checkout_str = checkout_date.strftime('%Y-%m-%d')
        url = (
            f"https://www.booking.com/searchresults.html?"
            f"ss=London&"
            f"group_adults=2&"
            f"no_rooms=1&"
            f"group_children=0&"
            f"checkin={checkin_str}&"
            f"checkout={checkout_str}&"
            f"selected_currency=USD"
        )
        print(f"Scraping for dates: {checkin_str} to {checkout_str}")
        driver = webdriver.Chrome()
        driver.get(url)
        time.sleep(10)
        try:
            close_button = WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, 'button[aria-label="Dismiss sign-in info."]'))
            )
            close_button.click()
        except TimeoutException:
            pass
        
        LOAD_MORE_XPATH = "//button[contains(., 'Load more results')]"
        max_loads = 100
        max_scroll_steps = 30
        for load_num in range(max_loads):
            load_more_btn = None
            for _ in range(max_scroll_steps):
                found = driver.find_elements(By.XPATH, LOAD_MORE_XPATH)  
                if found:
                    load_more_btn = found[0]
                    break
                driver.execute_script("window.scrollBy(0, 1000);")
                time.sleep(1)
            if load_more_btn is None:
                break
            driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", load_more_btn)
            time.sleep(0.5)
            driver.execute_script("arguments[0].click();", load_more_btn)
            
            time.sleep(3)
            
        time.sleep(2)
        
        hotel_cards = driver.find_elements(By.CSS_SELECTOR, '[data-testid="property-card"]')
        data = []
        for card in hotel_cards:
            data.append({
                'URL': get_data(card, '[data-testid="title-link"]', "href"),
                'Name': get_data(card, '[data-testid="title"]'),
                'Location': get_data(card, '[data-testid="address-link"]'),
                'Price': get_data(card, '[data-testid="price-and-discounted-price"]'),
                'Ratings': get_data(card, '[data-testid="review-score"]'),
                'Check_In_Date': checkin_date,
                'Scraped_At': datetime.now().isoformat(),
                'Search_URL': driver.current_url,
            })
        
        driver.quit()
        
        df = pd.DataFrame(data)
        df_cleaned = df.copy()
        df_cleaned = df_cleaned.drop_duplicates(subset=['URL'], keep='last')
        df_cleaned.columns = df_cleaned.columns.str.lower()
        df_cleaned['property_id'] = df_cleaned['url'].str.extract(r'hotel/[^/]+/([^./]+)')
        df_cleaned[['borough', 'city']] = df_cleaned['location'].str.split(', ', expand=True)
        mask = df_cleaned['city'].isna()
        df_cleaned.loc[mask, 'city'] = df_cleaned.loc[mask, 'borough'] 
        df_cleaned.loc[mask, 'borough'] = np.nan
        df_cleaned[['score_text', 'score', 'rating', 'reviews']] = df_cleaned['ratings'].str.split('\n', expand=True)
        df_cleaned['price'] = (df_cleaned['price'].str.replace(',', '', regex=False).str.findall(r'\d+(?:\.\d+)?').str[-1])
        df_cleaned['price'] = pd.to_numeric(df_cleaned['price'], errors='coerce')
        df_cleaned['score'] = df_cleaned['score'].astype(float)
        df_cleaned['reviews'] = df_cleaned['reviews'].str.replace(r'[^\d.]', '', regex=True).str.strip().astype('Int64')
        df_cleaned['check_in_date'] = pd.to_datetime(df_cleaned['check_in_date']).dt.date
        df_cleaned['check_in_weekday'] = pd.to_datetime(df_cleaned['check_in_date']).dt.day_name()
        score_mask = df_cleaned['score'] < 7
        df_cleaned.loc[score_mask, 'rating'] = pd.cut(df_cleaned.loc[score_mask, 'score'], bins=[0, 2.99, 4.99, 6.99], labels=['Very Poor', 'Poor', 'Average'])
        df_cleaned['reviews'] =  df_cleaned['reviews'].fillna(0)
        df_cleaned['score'] =  df_cleaned['score'].fillna(0)
        df_cleaned['rating'] =  df_cleaned['rating'].fillna('No Reviews')
        df_cleaned['borough'] =  df_cleaned['borough'].fillna('Unknown')
        df_cleaned = df_cleaned.drop(columns=['ratings', 'score_text', 'location'])
        
        server = r'(localdb)\MSSQLLocalDB'
        database = 'da_projects'
        db_driver = 'ODBC Driver 18 for SQL Server'
        connection_string = rf"mssql+pyodbc://@{server}/{database}?driver={db_driver}&trusted_connection=yes"
        engine = create_engine(connection_string, fast_executemany=True)
        
        table_name = 'bookings'
        df_cleaned.to_sql(
            name=table_name,
            con=engine, 
            if_exists='append', 
            index=False,
            schema='dbo'
        )
        
        file_name = 'cleaned_data.csv'
        file_exists = os.path.isfile(file_name)
        df_cleaned.to_csv(file_name, index=False, mode='a', header= not file_exists)

In [13]:
run(7)

Scraping for dates: 2026-10-13 to 2026-10-14
No more 'Load more results' button, stopping.
Scraping for dates: 2026-10-14 to 2026-10-15
No more 'Load more results' button, stopping.
Scraping for dates: 2026-10-15 to 2026-10-16
No more 'Load more results' button, stopping.
Scraping for dates: 2026-10-16 to 2026-10-17
No more 'Load more results' button, stopping.
Scraping for dates: 2026-10-17 to 2026-10-18
No more 'Load more results' button, stopping.
Scraping for dates: 2026-10-18 to 2026-10-19
No more 'Load more results' button, stopping.
Scraping for dates: 2026-10-19 to 2026-10-20
No more 'Load more results' button, stopping.
